<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-11-self-hosting/lesson-11.5-cloudrun-vs-gke/notebooks/GCP_Capstone_11.5_CloudRunVsGKE.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.5 Cloud Run or GKE Autopilot? — The Duty Cycle, Measured
**Netsetos GenAI Engineering — GCP Capstone** · Module 11 · rebuilt on the live lane, 10 September 2026

Module 11's climax, with one thing changed: the number the decision runs on is measured. Both bills are derived from the platforms' own price tables as before; the crossover is a duty cycle; the node-shape trap, the three ways the manifest could not start, the port-forward and the honest capability list stay word for word. New: the lane's own duty cycle from its usage rows (answers per day, GPU-seconds per answer from the self-hosted rows 11.4 wrote), the manifest parametrised from the project for the cluster gke.tf declares on the lane's VPC (`make gke-up` applies the workload, `make gke-down` removes it), the decision function run on the measured case, and the workload asserted absent as the last cell - because an L4 node left running bills by the hour, and the cluster's own fee is the shape's, not the hour's.

> **Nothing here needs a GPU.** The owner's hour (decision D5) runs from Cloud Shell; this notebook reads the manifest, prices it, and checks the workload is gone.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 pandas==2.3.3 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
GKE = False   # True: the owner's hour is running (make gke-up ... make gke-down, decision D5); False: the manifest is read and the workload asserted absent

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers):
    the body's `model` names what answered (a fallback included), the headers carry the cost the gateway priced."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def slm(path: str, body: dict | None = None, method: str = "POST", timeout: int = 240) -> tuple[int, dict | str, float]:
    """One call to the SLM's own doors (Ollama's /api/*, or its OpenAI-compatible /v1/*), timed - the first call after
    idle is the cold start."""
    t0 = time.time()
    r = requests.request(method, f"{SLM_URL}{path}", json=body, timeout=timeout,
                         headers={"Authorization": f"Bearer {documind_tools._id_token(SLM_URL)}"})
    try:
        return r.status_code, r.json(), time.time() - t0
    except ValueError:
        return r.status_code, r.text[:400], time.time() - t0

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They land in
# Cloud Logging first (the sink copies them into BigQuery for the view); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: run_eval, judge
sys.path.insert(0, f"{KIT}/deploy/services/slm")        # compare_backends, make_modelfile
print("helpers: api(), gateway(), slm(), service(), usage_rows(); the kit's evals/ and services/slm/ on sys.path")


## Cell 2: The customer 11.4 could not serve
Cloud Run's L4 in Mumbai is invitation-only. GKE's is not.


In [ ]:
# What 11.4 left open, and the customer it could not serve.
#
# You have a model on Cloud Run behind an L4. It scales to zero, costs nothing
# idle, and takes about 34 seconds to answer the first request after a nap.
#
# Two questions were left hanging:
#   1. Is a platform that keeps the pod warm cheaper, if you are busy enough?
#   2. What about the bank that needs the inference INSIDE India?
#
# The second one has a clean answer, and it is not the one you would guess.
PLATFORM_L4_IN_MUMBAI = {
    'Cloud Run':      ('asia-south1', 'INVITATION ONLY - you cannot self-serve it'),
    'GKE / Compute':  ('asia-south1', 'GENERALLY AVAILABLE - zones a, b and c'),
}
print(f'  {"platform":16} {"region":14} L4 availability')
for p, (r, s) in PLATFORM_L4_IN_MUMBAI.items():
    print(f'  {p:16} {r:14} {s}')
print()
print('  Same GPU, same region, two different answers - because availability is')
print('  a per-PRODUCT list, not a per-region one. 11.4 could not put an L4 in')
print('  Mumbai. GKE can, today, without asking anyone.')
print()
print('  That is the strongest reason in this lesson to choose GKE, and it has')
print('  nothing to do with cost or scale. Read the next three steps anyway -')
print('  the money still has to work.')


## Cell 3: What Cloud Run bills
Two rates, one of them zero - and the flags asserted from the kit's own deploy.


In [ ]:
# What Cloud Run actually bills, from 11.1's table.
USD_INR = 85
HOURS_PER_MONTH = 730          # Google's own convention: 365 x 24 / 12

CLOUD_RUN = [
    ('L4 GPU (no zonal redundancy)', 0.672),
    ('CPU (8 vCPU)',                 0.518),
    ('Memory (32 GiB)',              0.230),
]
cr_rate = sum(r for _, r in CLOUD_RUN)
for label, r in CLOUD_RUN:
    print(f'  {label:32} ${r:>8.3f}/hr')
print(f'  {"TOTAL, while serving":32} ${cr_rate:>8.3f}/hr')
print(f'  {"TOTAL, while idle":32} ${0:>8.3f}/hr   <- the whole point')
print()
print(f'  At 100% duty: ${cr_rate * HOURS_PER_MONTH:,.0f}/month = '
      f'Rs {cr_rate * HOURS_PER_MONTH * USD_INR:,.0f}')
print()
print('  Cloud Run bills per second of REQUEST HANDLING. A service nobody calls')
print('  costs nothing at all - and that zero is what GKE has to beat.')
# The rate that applies is the instance's, and the instance is the one the kit deploys: make deploy-slm asks for
# exactly these three lines. Asserted from the clone, so the table above is about the lane's own service.
mk = open(f"{KIT}/deploy/Makefile", encoding="utf-8").read()
DEPLOY_SLM = mk.split("deploy-slm:", 1)[1].split(chr(10) * 2, 1)[0]
assert "--gpu 1 --gpu-type nvidia-l4 --no-gpu-zonal-redundancy" in DEPLOY_SLM and "--cpu 8 --memory 32Gi" in DEPLOY_SLM
print("\n(make deploy-slm: --gpu 1 --gpu-type nvidia-l4 --no-gpu-zonal-redundancy --cpu 8 --memory 32Gi - the three rows above)")


## Cell 4: What Autopilot bills
Node + three premiums + a cluster fee. Four lines; people quote the first.


In [ ]:
# What GKE Autopilot bills, which is a different shape entirely.
#
# Autopilot does not price "a GPU". It prices the NODE your pod lands on, adds
# a premium per accelerator, per vCPU and per GiB, and charges a flat cluster
# fee on top. Four lines, and people quote the first one.
G2_STANDARD_8  = 0.853624      # 8 vCPU, 32 GiB, 1 x L4
PREM_GPU       = 0.067         # per GPU-hour
PREM_VCPU      = 0.003         # per vCPU-hour
PREM_MEM       = 0.00035       # per GiB-hour
CLUSTER_FEE    = 0.10          # per cluster-hour, after the free-tier credit

LINES = [
    ('g2-standard-8 node',        G2_STANDARD_8),
    ('L4 accelerator premium',    PREM_GPU),
    ('vCPU premium (8)',          8 * PREM_VCPU),
    ('memory premium (32 GiB)',   32 * PREM_MEM),
]
pod = sum(r for _, r in LINES)
for label, r in LINES:
    print(f'  {label:32} ${r:>8.4f}/hr')
print(f'  {"POD SUBTOTAL":32} ${pod:>8.4f}/hr')
print(f'  {"Autopilot cluster fee":32} ${CLUSTER_FEE:>8.4f}/hr')
gke_rate = pod + CLUSTER_FEE
print(f'  {"ALL-IN":32} ${gke_rate:>8.4f}/hr')
print()
print(f'  ${gke_rate * HOURS_PER_MONTH:,.0f}/month = '
      f'Rs {gke_rate * HOURS_PER_MONTH * USD_INR:,.0f}, running or not.')
print()
print('  The cluster fee is the line people forget. It is charged per CLUSTER,')
print('  not per pod, so it is nearly free if you run twenty services on one')
print('  cluster and it is 9% of the bill if you run exactly one - which is')
print('  what this comparison does, and what a demo cluster always does.')


## Cell 5: The break-even duty cycle


In [ ]:
# The only question that matters: how busy are you?
#
# Cloud Run charges while it serves. GKE charges while it exists. So the answer
# is a DUTY CYCLE, not a preference.
def monthly_inr(cloud_run_duty: float) -> tuple[float, float]:
    cr = cr_rate * HOURS_PER_MONTH * cloud_run_duty * USD_INR
    gke = gke_rate * HOURS_PER_MONTH * USD_INR          # always on
    return cr, gke


breakeven = gke_rate / cr_rate
print(f'  BREAK-EVEN DUTY CYCLE: {breakeven:.1%} '
      f'({breakeven * HOURS_PER_MONTH:,.0f} of {HOURS_PER_MONTH} hours a month)')
print()
print(f'  {"duty cycle":>12} {"hours/mo":>10} {"Cloud Run":>12} {"GKE":>12}  cheaper')
for duty in (0.02, 0.10, 0.33, 0.50, breakeven, 0.90, 1.00):
    cr, gke = monthly_inr(duty)
    print(f'  {duty:>11.0%} {duty * HOURS_PER_MONTH:>10,.0f} {cr:>12,.0f} {gke:>12,.0f}  '
          f'{"GKE" if gke < cr * 0.999 else "Cloud Run" if cr < gke * 0.999 else "identical"}')
print()
print('  Read the top rows. A service handling a few thousand questions a day is')
print('  busy for maybe 2% of the month - and there Cloud Run is FIFTY TIMES')
print('  cheaper. GKE only wins when the pod is genuinely working three quarters')
print('  of every hour of every day, which almost nothing in a document system')
print('  does. Measure your own duty cycle before you migrate anything.')


## Cell 6: The duty cycle, measured
Answers per day and GPU-seconds per answer from the lane's usage rows.


In [ ]:
import math

# THE DUTY CYCLE, MEASURED. The first version ran its decision on "DocuMind chat today (bursty, 2%)" - a typed
# constant. The lane has the number: answers per day from the usage rows, and GPU-seconds per answer from the
# latency_ms of the rows the self-hosted route answered (model_backend=gateway, model documind-slm; 11.4's pin and
# candidate wrote some). Where no such row exists in the last day the estimate stays, and is labelled as one.
rows = usage_rows(minutes=60 * 24, limit=1000)
slm_rows = [r for r in rows if r.get("model_backend") == "gateway" and "slm" in str(r.get("model", ""))]
ANSWERS_PER_DAY = len(rows)
SECONDS_PER_ANSWER = (sum(r["latency_ms"] for r in slm_rows) / len(slm_rows) / 1000) if slm_rows else 2.5
label = f"measured over {len(slm_rows)} self-hosted rows" if slm_rows else "11.4's estimate: no self-hosted row in the last day"
print(f"the lane, last 24 h: {ANSWERS_PER_DAY} answers; {SECONDS_PER_ANSWER:.2f} GPU-seconds per answer ({label})")

def verdict(answers: int, seconds: float = SECONDS_PER_ANSWER) -> tuple[float, float, float]:
    """(load, Cloud Run Rs, GKE Rs) for a month. Cloud Run bills instance-seconds, so this works above 100% too."""
    gpu_seconds = answers * seconds * 30
    load = gpu_seconds / (HOURS_PER_MONTH * 3600)
    return load, cr_rate * HOURS_PER_MONTH * load * USD_INR, gke_rate * HOURS_PER_MONTH * USD_INR

print(f"\n{'answers/day':>12} {'load':>7} {'Cloud Run Rs':>13} {'GKE Rs':>10}  cheaper")
for n, tag in [(max(ANSWERS_PER_DAY, 1), "  <- the lane today"), (500, ""), (5_000, ""), (26_000, ""), (50_000, ""), (200_000, "")]:
    load, cr, gke = verdict(n)
    note = "" if load <= 1 else f"  needs {math.ceil(load)} concurrent instances (11.4's --max-instances 1 caps it)"
    print(f"{n:>12,} {load:>7.1%} {cr:>13,.0f} {gke:>10,.0f}  {'GKE' if gke < cr else 'Cloud Run'}{tag}{note}")
LANE_DUTY = verdict(max(ANSWERS_PER_DAY, 1))[0]
crossover = breakeven * HOURS_PER_MONTH * 3600 / (30 * SECONDS_PER_ANSWER)
print(f"\nthe lane's duty cycle is {LANE_DUTY:.2%}; the crossover is about {crossover:,.0f} answers a day at {SECONDS_PER_ANSWER:.1f} s each")


## Cell 7: The resource request that buys a bigger node, and three ways the manifest could not start


In [ ]:
# The resource request that quietly buys a bigger node.
#
# Autopilot picks the node from what your pod ASKS FOR. A pod that asks for a
# whole machine leaves nothing for the kubelet and system DaemonSets, so
# Autopilot does not fail - it rounds UP to the next shape, and you pay for
# capacity you never requested and cannot use.
G2_STANDARD_12 = 1.000416      # 12 vCPU, 48 GiB, 1 x L4

def all_in(node_rate: float, vcpu: int, gib: int) -> float:
    return node_rate + PREM_GPU + vcpu * PREM_VCPU + gib * PREM_MEM + CLUSTER_FEE


SHAPES = [
    ('asks cpu 6 / 24Gi  -> g2-standard-8',  all_in(G2_STANDARD_8, 8, 32)),
    ('asks cpu 8 / 32Gi  -> g2-standard-12', all_in(G2_STANDARD_12, 12, 48)),
]
for label, rate in SHAPES:
    print(f'  {label:38} ${rate:.4f}/hr  Rs {rate * HOURS_PER_MONTH * USD_INR:>8,.0f}/mo  '
          f'break-even {rate / cr_rate:.0%}')
print()
delta = SHAPES[1][1] - SHAPES[0][1]
print(f'  Asking for two more vCPU you will not use costs Rs '
      f'{delta * HOURS_PER_MONTH * USD_INR:,.0f} a month')
print(f'  and moves the break-even from {SHAPES[0][1] / cr_rate:.0%} to '
      f'{SHAPES[1][1] / cr_rate:.0%} - which is the difference between')
print('  "GKE might be worth it" and "GKE is never worth it".')
print()
print('  The manifest in this repo asked for exactly 8 / 32Gi until 5 September 2026.')
print('  It now asks for 6 / 24Gi, and the comment says why.')


In [ ]:
# The manifest, and the three ways it could not start.
#
# Every one of these was in the shipped file until 5 September 2026. None of them would
# have produced a useful error.
FIXED = [
    ('stock image + HF_HUB_OFFLINE=1',
     'ran upstream vllm/vllm-openai, asked for google/gemma-3-4b-it, then '
     'blocked the only way to fetch it. Now runs the image 11.1 BUILDS, which '
     'has the weights baked - so offline is finally correct.'),
    ('cpu 8 / memory 32Gi',
     'exactly g2-standard-8. Rounded up to -12. Now 6 / 24Gi.'),
    ('"behind the GKE Inference Gateway"',
     'no Gateway, InferencePool or HTTPRoute exists in this repo. Comment removed '
     'rather than left to be believed.'),
]
for what, why in FIXED:
    print(f'  {what}')
    print(f'     {why}')
    print()
print('  The pattern is the same each time: the manifest was WELL-FORMED and')
print('  WRONG. kubectl accepts all three happily. Two of them fail at runtime')
print('  in ways that look like a slow cluster, and the third fails on the')
print('  invoice a month later.')


## Cell 8: The manifest, parametrised
The image from the project, the cluster on the lane's VPC, the hour printed for Cloud Shell.


In [ ]:
# THE MANIFEST, PARAMETRISED. deploy/gke/vllm-deployment.yaml runs the image 11.1 BUILDS (weights baked, so
# HF_HUB_OFFLINE=1 is correct), asks for 6 vCPU / 24 GiB rather than the whole node, and its image line is ${IMAGE}:
# make gke-up fills it from the lane's own repository with envsubst - until Module 11 it named a placeholder project,
# and kubectl pulled a tag that could not exist. gke.tf declares the cluster on the lane's VPC - part of the one shape since
# 15 September 2026, created by make up; the two targets below touch only the workload.
# The Service is ClusterIP, reached with a port-forward, never a load balancer. Terraform and kubectl live in Cloud
# Shell, not here: the hour is the owner's (decision D5), and this cell prints it.
manifest = open(f"{KIT}/deploy/gke/vllm-deployment.yaml", encoding="utf-8").read()
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/documind/gemma-vllm:latest"
assert "image: ${IMAGE}" in manifest and 'cpu: "6"' in manifest and "memory: 24Gi" in manifest and "type: LoadBalancer" not in manifest
rendered = manifest.replace("${IMAGE}", IMAGE)
print(chr(10).join(l for l in rendered.splitlines() if l.strip().startswith(("image:", "cpu:", "memory:", "nvidia.com/gpu", "cloud.google.com/gke-accelerator", "name: HF_HUB", "path: /health"))))
tf = open(f"{KIT}/deploy/terraform/gke.tf", encoding="utf-8").read()
assert "google_container_cluster" in tf and "var.gke_cluster" not in tf and "google_compute_network.vpc.id" in tf and "google_compute_subnetwork.subnet.id" in tf
print()
print(chr(10).join(l for l in mk.split("gke-up:", 1)[1].split(chr(10) * 2, 1)[0].splitlines()[:5]))
print(f"""
make gke-up PROJECT={PROJECT_ID}                 # the manifest applied on documind-autopilot (gke.tf's cluster); an L4 node takes minutes, not seconds
kubectl get pods -w
kubectl port-forward svc/documind-vllm 8080:80 &
curl -s localhost:8080/v1/chat/completions -H "Content-Type: application/json" -d '{{"model":"google/gemma-3-4b-it","messages":[{{"role":"user","content":"What is the notice period?"}}]}}'
make gke-down PROJECT={PROJECT_ID}               # the workload and its node; the cluster stays (make down removes it - its fee is the shape's)""")
if GKE:
    print(subprocess.run(["gcloud", "container", "clusters", "list", "--project", PROJECT_ID, "--format=table(name,location,status,currentNodeCount)"],
                         capture_output=True, text=True).stdout)


## Cell 9: What GKE actually gives you, honestly - and the decision as a function


In [ ]:
# What GKE gives you that Cloud Run does not - and what it costs to wire.
#
# Be honest about this, because it is the usual reason people pick Kubernetes
# and the kit does not currently deliver it.
HAVE = [
    ('warm pod, no cold start',  'YES', 'the model stays in GPU memory'),
    ('choose the node shape',    'YES', 'and the region, including Mumbai'),
    ('run 20 services per GPU cluster', 'YES', 'amortises the $0.10/hr fee'),
    ('scale on request queue depth', 'NOT YET',
     'needs Managed Prometheus + a custom-metrics adapter'),
    ('scale to zero',            'NO',
     'that is Cloud Run\'s trick; KEDA can approximate it, at complexity'),
]
print(f'  {"capability":34} {"kit":>8}  note')
for cap, state, note in HAVE:
    print(f'  {cap:34} {state:>8}  {note}')
print()
print('  The queue-depth HPA is the one worth wanting: vLLM exposes')
print('  `vllm:num_requests_waiting`, and scaling on QUEUE rather than CPU is')
print('  the correct signal for an inference server - CPU stays flat while')
print('  requests pile up behind a busy GPU.')
print()
print('  It needs Managed Prometheus scraping the pod and a custom-metrics')
print('  adapter feeding the HPA. That is three more manifests this kit does')
print('  not provision, so this lesson explains the wiring instead of shipping')
print('  a manifest nobody has run. An HPA you cannot demonstrate is a')
print('  screenshot, not a lesson.')


In [ ]:
# THE DECISION, AS A FUNCTION - with the measured case in it. Residency and a cold-start ban decide before cost
# does; then the break-even, with only the cluster fee shared across services on a cluster (each service still needs
# its own GPU node). The first case is the lane's, from Cell 6; the other four are the hypotheticals kept from the
# first version, so the room can see which inputs move the answer.
def choose(duty_cycle: float, needs_india: bool, services_on_cluster: int = 1, can_tolerate_cold_start: bool = True) -> tuple[str, str]:
    """Which platform, and the one-line reason you would give in a review."""
    if needs_india:
        return ("GKE", "Cloud Run L4 in asia-south1 is invitation-only; GKE G2 is GA there")
    if not can_tolerate_cold_start:
        return ("GKE", "a warm pod has no 34-second first request")
    pod_only = gke_rate - CLUSTER_FEE
    effective = pod_only + CLUSTER_FEE / services_on_cluster
    if duty_cycle > effective / cr_rate:
        return ("GKE", f"above the {effective / cr_rate:.0%} break-even at this duty cycle")
    return ("Cloud Run", "scale-to-zero wins below the break-even, and it is simpler")

CASES = [
    (f"DocuMind today (measured: {LANE_DUTY:.2%})", LANE_DUTY, False, 1, True),
    ("the bank: inference must be in India", LANE_DUTY, True, 1, True),
    ("batch ingest, 20 hours a day", 0.83, False, 1, True),
    ("interactive demo, no cold start", 0.10, False, 1, False),
    ("platform team, 8 services on one cluster", 0.15, False, 8, True),
]
print(f"  {'case':44} {'pick':>10}  why")
for label, duty, india, n, cold in CASES:
    pick, why = choose(duty, india, n, cold)
    print(f"  {label:44} {pick:>10}  {why}")
assert choose(LANE_DUTY, False)[0] == "Cloud Run", "the lane's own duty cycle crossed the break-even: re-read Cell 6 before believing it"
print("\n  four of five say Cloud Run, and the first is measured. GKE wins on RESIDENCY and on genuinely steady load - not on a preference.")


## Cell 10: Delete it
The gate: the cluster is absent.


In [ ]:
# DELETE IT. The gate for the lesson: the workload is gone. Deleting the pod releases the GPU node and most of the bill;
# the cluster itself is Terraform's (gke.tf) and charges its fee whether or not a pod runs - about Rs 6,200 a month,
# the shape's line, not this lesson's. After the owner's hour, make gke-down is what makes the assertion true again.
clusters = subprocess.run(["gcloud", "container", "clusters", "list", "--project", PROJECT_ID, "--format=value(name,location,status)"],
                          capture_output=True, text=True).stdout.split()
print("clusters:", clusters or "none")
for label, rate in (("cluster + pod left running", gke_rate), ("pod deleted, cluster kept", CLUSTER_FEE), ("cluster deleted", 0.0)):
    print(f"  {label:30} Rs {rate * HOURS_PER_MONTH * USD_INR:>9,.0f}/month")
if GKE:
    print(f"\nthe hour is running: make gke-down PROJECT={PROJECT_ID} when it ends, then re-run this cell with GKE = False")
else:
    subprocess.run(["gcloud", "container", "clusters", "get-credentials", "documind-autopilot", "--region", REGION, "--project", PROJECT_ID],
                   capture_output=True, text=True)
    r = subprocess.run(["kubectl", "get", "deployment", "documind-vllm", "-o", "name"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        raise AssertionError(f"the vLLM workload is still deployed: make gke-down PROJECT={PROJECT_ID}")
    print("\ndocumind-vllm: no deployment on documind-autopilot" + ("" if r.returncode == 0 else " (kubectl could not ask; `kubectl get deployment documind-vllm` from Cloud Shell)"))
    print("11.4 said forgetting min-instances 0 was the dearest mistake; a GPU node left under a pod is the same mistake without scale-to-zero to save you.")


## Cell 11: Look how far


In [ ]:
# Module 11, end to end - on the lane.
STAGES = [
    ("11.1", "the bill before the build; the quota read; the kit's image and deploy; one completion through an OpenAI-compatible door"),
    ("11.2", "the server as the kit's seven files, two doors, the SSE frames asserted, guided JSON on the lane's text"),
    ("11.3", "the gateway as a route: the config, the classifier proven, the hook run, the real gateway called"),
    ("11.4", "the GPU as a bill: 10.5's model deployed, a candidate on MODEL_BACKEND=gateway judged by the gate, one tenant pinned, slm-off asserted"),
    ("11.5", "the duty cycle measured, the manifest parametrised, the decision as a function, the workload asserted absent"),
]
for lid, what in STAGES:
    print(f"  {lid}  {what}")
print()
print("  every lesson answered a HOSTING question with a NUMBER - and this time the number came from the lane, not from a slide.")
print("  the three that decided the last two: the instance price, the duty cycle, and a region list.")


## Done — Module 11 is complete

Both bills derived, the crossover at about 74% duty cycle, the lane's own duty cycle measured and far below it, the two-vCPU over-request that moves the crossover to 86%, three ways a well-formed manifest fails silently, a ClusterIP reached without a load balancer, the capability list without a screenshot in it, the decision as a function with the measured case first, and the cluster asserted absent.

**The answer for DocuMind is Cloud Run** - on cost, at the lane's measured volume and at any realistic one. **The answer for the bank is GKE**, because an L4 in Mumbai exists on one platform and not the other.

**Next:** Module **12** ships it - keyless CI/CD behind an eval gate in **12.7**, every surface behind IAP in **12.8**. Then Module **13** asks you to defend these choices out loud, and "we preferred Kubernetes" is not an answer that survives a rubric.
